In [6]:
import pandas as pd
import numpy as np

def reduce_mem_usage(df):
    """ Iterate through all columns and modify the data type to reduce memory usage. """
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float32) # float16 ist oft langsam, float32 reicht
                else:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage decreased to {end_mem:.2f} MB')
    return df

# JETZT DER TRICK: Wir laden erst mal nur die ersten 100.000 Zeilen zum Testen 
# ODER wir laden alles, aber rufen sofort die Funktion auf.
# Da du einen MemoryError hast, probieren wir es hiermit:

print("Loading Transaction Data...")
train_transaction = pd.read_csv('../data/train_transaction.csv')
train_transaction = reduce_mem_usage(train_transaction)

print("Loading Identity Data...")
train_identity = pd.read_csv('../data/train_identity.csv')
train_identity = reduce_mem_usage(train_identity)

# Merge erst nachdem beide verkleinert wurden
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
print(f"Final Shape: {train.shape}")

Loading Transaction Data...
Memory usage decreased to 916.30 MB
Loading Identity Data...
Memory usage decreased to 31.91 MB
Final Shape: (590540, 434)
